<a href="https://www.kaggle.com/code/samratrm/gemstone-linear-regression?scriptVersionId=325363161" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Gemstone Price Prediction using Linear Regression

This notebook is designed for learning and hands-on practice with a complete machine learning workflow. It is beginner-friendly and explains each concept step by step.

In this project, we work with a gemstone dataset and go through the entire data science process, including:

* Data Understanding – Exploring the dataset, understanding features, and identifying the target variable.
* Data Cleaning – Handling missing values, duplicates, incorrect data types, and other data quality issues.
* Exploratory Data Analysis (EDA) – Visualizing data distributions, discovering patterns, relationships, and insights using charts and statistical summaries.
* Feature Engineering – Creating, transforming, and selecting features to improve model performance.
* Data Preprocessing – Encoding categorical variables, scaling numerical features, and preparing data for machine learning algorithms.
* Model Building – Training a Linear Regression model to predict gemstone prices.
* Model Evaluation – Measuring performance using appropriate evaluation metrics and interpreting the results.
* Key Learnings and Insights – Understanding how each step contributes to building an effective machine learning model.

Every stage is explained in simple language with detailed comments and examples, making this notebook suitable for beginners who want to understand not only how to build a machine learning model, but also why each step is important.

### Initial Setup 

In [ ]:
import numpy as np
import pandas as pd
import kagglehub
import matplotlib.pyplot as plt 
import seaborn as sns 
import os

import sklearn

In [ ]:
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
train = pd.read_csv("/kaggle/input/competitions/playground-series-s3e8/train.csv")
test = pd.read_csv("/kaggle/input/competitions/playground-series-s3e8/test.csv")
sample = pd.read_csv("/kaggle/input/competitions/playground-series-s3e8/sample_submission.csv")

In [ ]:
train.head()

## Data Understanding

### Data Description 

`id` — your uniqur row ID

`carat` — the diamond's weight. Carat refers to the unit of weight measurement used exclusively to weigh gemstones and diamonds. This is usually the single strongest predictor of price.

`cut` — quality of the cut (how well-proportioned it is). An ordered quality grade, typically Fair < Good < Very Good < Premium < Ideal.

`color` — diamond color grade, lettered D (best, colorless) through J (more tinted). Counterintuitively, *earlier* letters are better.

`clarity` — how free of internal flaws it is. Clarity is a measure of the purity and rarity of the stone, graded by the visibility of characteristics under 10-power magnification. Ordered worst-to-best roughly: I1 < SI2 < SI1 < VS2 < VS1 < VVS2 < VVS1 < IF.

`depth` — the diamond's height as a percentage. It's the height in millimeters measured from the culet (bottom tip) to the table (flat top surface), expressed relative to width.

`table` — the facet you see when the stone is viewed face-up, as a percentage of the diamond's width.

`x`, `y`, `z` — the physical dimensions in millimeters (length, width, depth). These are where you'll spot the `0` values that are physically impossible — the data-cleaning catch I mentioned.

`price` — your target, in US dollars.

In [ ]:
test.head()

The target variable is the `price` column. 


#### Initial Intuition 

- The price of a gemstone depends on the carat, size and scarcity.
    - Large gemstones are rare and are more expensive. So, higher the carat the price increases non linearly.
    - And the size of the stone matters as much as carat. 
    - The scarcity is not a factor in our dataset. But its hidden in other features.
        -  Big, flawless, colorless stones are exactly what's rare. 
- The cut, colour and clarity is also a factor.
    - Cut refers to how well the gemstone is shaped and polished. Better cuts are sparkels more.
    - Colour is also a importnat factor (D-J grade). (image attached for reference)
    - Clarity refers to imperfections on the surface of the gemstone. Fewer imperfections means more value.

-> We can see after the regression what and how much importance our model gives to each features.

In [ ]:
train['color'].value_counts()

In [ ]:
sample.head()

In [ ]:
train.info()

We might have to convert `cut` , `color` and `clarity` into categorical data. Since, we can't input string data in the ML model. 

In [ ]:
train.describe()

There is no `nan` data which we can see in the count. But there might be some outliars and bad data which we will remove in data cleaning. Like the `x`,`y`,`z` being 0 which is bad because height, width and depth can't be 0. 

## Data Cleaning 

Delete the `id` column. Since it can't use as a input to train the model. Its not a feature.

In [ ]:
train = train.drop("id",axis=1)
test = test.drop("id",axis=1)

train.head()

Now, Lets delete all the impossible rows (bad data)

Like the rows that are 0 for measurements like length, width and depth

In [ ]:
train.isna().sum()

There is no rows with `nan` values.

In [ ]:
(train[['x','y','z']] == 0).any(axis=1).sum()

There are 10 rows that have atleast one data point or row (x,y,z) as 0

In [ ]:
index = (train[['x','y','z']] != 0).all(axis=1)
train = train[index]
train.head()

In [ ]:
(train[['x','y','z']] == 0).any(axis=1).sum()

All the x,y,z data points that had atleast one value as 0 are deleted.

In [ ]:
train.duplicated().sum()

There is no duplicate row.

In [ ]:
train.describe()

We have removed all the nan and impossible data from the x,y,z rows. Now lets work on the outliars. 

In [ ]:
train[['carat','depth','table','x','y','z']].boxplot()
plt.show()

In [ ]:
train[train['z']>train['x']].shape

So, there are 3 stones that have depth higher than the length. 

Higher depth means bigger size means higher cost. So, Lets skip removing the max side outliars for now. We can focus on this after out firt model evaluation. 

In [ ]:
train.info()

Columns `clarity` , `cut` , `colour` are all categorical data. So, lets change the datatypes. 

In [ ]:
cat_cols = train.select_dtypes(include='object').columns
train[cat_cols] = train[cat_cols].astype('category')
test[cat_cols] = test[cat_cols].astype('category')
train.info()

In [ ]:
test.info()

### Data Cleaning Summary:

- Removed the `id` column.
- Checked for `nan` and duplicate rows. No data found
- Removed the (min side outliars) impossible values for x,y,z columns which was removing all the rows which had 0.
- The max side outliars are not removed. It will be worked on after the initial model evaluation.
- Changed dtype for categorical data.

## EDA 

In [ ]:
train['price'].hist(bins=30)

In [ ]:
train['price'].skew()

The data is heavily right skewed. Linear regression assumes that errors are roughly normally distributed and have constant variance. With a heavily skewed target, large values dominate the loss function because squared errors explode.

We should do `np.log1p()` to make it more normally distributed. So, the model learns relative differences rather than absolute differences. 

We do this in the Feature engineering section to avoid disturbing the other price related EDA. 

In [ ]:
train.corr(numeric_only=True)['price'].sort_values(ascending=False)

Note: Remember the depth is a percent realtive to width. And Z is the measurement of depth. 

- Size dominates. x,y,z are very strongly related to price.
- Weight also matters as much as size. Its also very strongly related.
- Table and depth are weak but we will keep them and let the model regularization decide the importance. 

In [ ]:
features = ['x', 'y', 'z', 'carat']

fig, axes = plt.subplots(2,2,figsize=(12,10))

for ax, col in zip(axes.flat, features):
    ax.scatter(train[col], train['price'], s=3, alpha=0.3)
    ax.set_xlabel(col)
    ax.set_ylabel('price')
    ax.set_title(f'price vs {col}')

plt.tight_layout()
plt.show()

there is just few very clear outliar in `z` we will remove it. 

In [ ]:
train[train['z']>7]

In [ ]:
train = train.drop(train[train['z']>7].index)

In [ ]:
features = ['x', 'y', 'z', 'carat','depth','table']

fig, axes = plt.subplots(3,2,figsize=(12,14))

for ax, col in zip(axes.flat, features):
    ax.scatter(train[col], train['price'], s=3, alpha=0.3)
    ax.set_xlabel(col)
    ax.set_ylabel('price')
    ax.set_title(f'price vs {col}')

plt.tight_layout()
plt.show()

These plots show the relationship of the features more clearly

In [ ]:
corr_matrix = train[['carat','x','y','z']].corr()

In [ ]:
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.show()

All 4 features are multicollinear to each other.

multicollinearity does not hurt your model's predictions or its score. The model still predicts price perfectly well with all four columns. What it breaks is coefficient interpretation — when x, y, z, carat are 0.98 correlated, the model can't tell which one deserves credit, so it splits the weight between them arbitrarily, and you'll see coefficients that look huge, tiny, or even negative in ways that make no physical sense. 

So if your only goal were leaderboard score, you could technically leave all four in and be fine or deal with this in feature enginerring section. 

 multicollinearity is a coefficient-interpretation problem here, not a prediction problem. 

In [ ]:
train.groupby('color', observed=False)['price'].mean().sort_values()

In [ ]:
mean_price = train.groupby('color', observed=True)['price'].mean().sort_values()

plt.figure(figsize=(8,4))
sns.barplot(
    x=mean_price.index,
    y=mean_price.values
)
plt.xlabel('color')
plt.ylabel('Mean Price')
plt.title('Average Price by Color')
plt.show()

D → Completely colorless <Br>
E → Nearly colorless<Br>
F → Nearly colorless<Br>
G → Slight trace of color<Br>
H → Slight trace of color<Br>
I → Noticeable warmth<Br>
J → More noticeable yellow/brown tint<Br>

D < E < F < G < H < I < J

The training data follows the order of the colour. 

In [ ]:
train.groupby('clarity', observed=False)['price'].mean().sort_values()

In [ ]:
mean_price = train.groupby('clarity', observed=True)['price'].mean().sort_values()

plt.figure(figsize=(8,4))
sns.barplot(
    x=mean_price.index,
    y=mean_price.values,
    order=mean_price.index
)
plt.xlabel('clarity')
plt.ylabel('Mean Price')
plt.title('Average Price by Clarity')
plt.show()

Actual order: IF > VVS1 > VVS2 > VS1 > VS2 > SI1 > SI2 > I1

#### Observation 

The order of the clarity is wrong. The best clarity grades sit at the bottom — VVS1 averages ~$2,013, IF ~$2,127 — while the worst grade SI2 tops the list at ~$5,219. So on average, worse clarity = higher price. Backwards from intuition, exactly the confound.
The mechanism, restated in dollars: SI2 diamonds cost more on average not because flaws are valuable, but because SI2 stones tend to be big. 

So the conclusion holds firmly: clarity looks worthless-or-backwards on its own, but that's only because size is confounding it. In your regression, with carat and clarity both present, the model will recover the true effect among same-size diamonds, better clarity adds value. 

In [ ]:
mean_price = train.groupby('cut', observed=True)['price'].mean().sort_values()

plt.figure(figsize=(8,4))
sns.barplot(
    x=mean_price.index,
    y=mean_price.values,
    order=mean_price.index
)
plt.xlabel('cut')
plt.ylabel('Mean Price')
plt.title('Average Price by cut')
plt.show()

Cut is also similar to the clarity. Cut also is not in the order we expected to the price but cut makes more sense when we analyse it with size features.

### Summary of EDA:

- We found multicollinearity in 4 featurres x,y,z and carat.
- We removed some extreme outliars from the z col to make the plot more viz.
- We discovered that colour feature follows the price trends of real world but clarity does not beaucse of size.
- We found that price is right skewed and needs to be normalized.
- The carat and x,y,z size parameters are non linear relationship with the price. So, we will be using the ploynomial regresion.

In [ ]:
train.head()

## Feature Engineering 

In [ ]:
train['volume'] = train['x'] * train['y'] * train['z']
test['volume'] = test['x'] * test['y'] * test['z']
train.head()

In [ ]:
train['density'] = train['carat'] / (train['volume'] + 1e-6)
test['density'] = test['carat'] / (test['volume'] + 1e-6)
train.head()

In [ ]:
mask = (train['density'] < 0.010)

plt.scatter(train.loc[mask,'density'],train.loc[mask,'price'])
plt.xlabel("Density")
plt.ylabel("Price")
plt.title("Density vs Price")
plt.show()

We have introduced a new features density and volume. Density is almost constant but we experiement with it before removing it since there might be some hidden relationship with other featuers. We can experiment with the model trainig to decide if we want to keep the x,y,z or just volume and similarly for density.

## Data Processing 

#### Encoding 

Since the colour, clarity and cut is ordinal we do Ordinal Encoding to keep the features meaningful. 

In [ ]:
num_feats = ["carat",'depth','table','x','y','z','volume'] # default 
cat_feats = ['cut','color','clarity']

In [ ]:
X = train[num_feats + cat_feats]
X.describe()

In [ ]:
y = train['price']
y.describe()

In [ ]:
y.hist(bins=30)

In [ ]:
y_log = np.log1p(y)
y_log.hist(bins=30)


#### Why ? 

`log1p` doesn't make the data "relative" — what it changes is **what counts as an equal-sized error.**

On the raw price scale, errors are absolute: being off by 500 is "the same mistake" whether the diamond costs 1,000 or 50,000.

On the log scale, equal distances are equal *ratios*, not equal dollar amounts. log(1000)→log(2000) is the same step as log(20000)→log(40000) — both are "doubling." So a fixed error in log space means a fixed *percentage* error in price.

Why: logs turn multiplication into addition. log(a×k) = log(a) + log(k). So multiplying price by 2 always adds the same constant, no matter the starting price. That's the whole mechanism — ratios become equal steps.

Why it helps here: diamond price spans ~$300 to ~$19,000. Without the log, the model obsesses over the few expensive stones (a 10% error there is huge in dollars) and neglects cheap ones. On the log scale it cares about *percentage* accuracy evenly across the range — which is usually what you actually want, and it tames the skew at the same time.

(The `1p` part = `log(1 + x)`, just so `x=0` doesn't blow up. Undo with `expm1`.)

In [ ]:
from sklearn.preprocessing import StandardScaler, PolynomialFeatures, OneHotEncoder
from sklearn.pipeline import Pipeline 
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression 
from sklearn.model_selection import cross_val_predict, cross_val_score
from sklearn.metrics import mean_squared_error
import warnings

In [ ]:
num_pipeline = Pipeline([
    ('poly',PolynomialFeatures(degree=2,include_bias=False)),
    ('scale',StandardScaler())])

pre_process_pipeline = ColumnTransformer([
    ('cat',OneHotEncoder(handle_unknown='ignore'),cat_feats),
    ('num',num_pipeline,num_feats)])

pipe = Pipeline([('pre',pre_process_pipeline),('model',LinearRegression())])

Note: The whole trick of polynomial regression: you don't make the model nonlinear, you feed it nonlinear features and let it stay linear.

## Model Building 

In [ ]:
X[num_feats].replace([np.inf, -np.inf], np.nan).isna().sum()

In [ ]:
pred_log = cross_val_predict(pipe,X,y_log,cv=5)
pred_price = np.expm1(pred_log)
rmse = np.sqrt(mean_squared_error(train['price'], pred_price))

print(rmse)

This is the baseline score we use this to compare new features and improve the model and scores.

Now lets include all the new featues and check the score. 

In [ ]:
len(sample) , len(test)

In [ ]:
m = pipe.fit(X,y_log)
X_test = test[num_feats+cat_feats]
pred_log = m.predict(X_test)
sample['price'] = np.expm1(pred_log)
sample.to_csv('submission.csv',index=False)

In [ ]:
print([f for f in os.listdir() if f.endswith('.csv')])  

In [ ]:
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.exceptions import ConvergenceWarning

# Ignore specific ConvergenceWarnings
warnings.filterwarnings("ignore", category=ConvergenceWarning)
for name, model in [('OLS',LinearRegression()),
                    ('Ridge',Ridge(alpha=1.0)),
                    ('Lasso',Lasso(alpha=0.001)),
                    ('Elastic',ElasticNet(alpha=0.001,l1_ratio=0.05))] : 
    pipe = pipe = Pipeline([('pre',pre_process_pipeline),('model',model)])
    pred_log = cross_val_predict(pipe,X,y_log,cv=5)
    pred_price = np.expm1(pred_log)
    rmse = np.sqrt(mean_squared_error(train['price'], pred_price))
    
    print(name , rmse)

In [ ]:
warnings.filterwarnings("ignore", category=ConvergenceWarning)
pipe = pipe = Pipeline([('pre',pre_process_pipeline),('model',ElasticNet(alpha=0.001,l1_ratio=0.05))])
pred_log = cross_val_predict(pipe,X,y_log,cv=5)
pred_price = np.expm1(pred_log)
rmse = np.sqrt(mean_squared_error(train['price'], pred_price))
print(rmse)

In [ ]:
num_feats_1 = ["carat",'depth','table','x','y','z','volume','density']
X1 = train[num_feats_1 + cat_feats].copy()
X1.head()

In [ ]:
pipe = Pipeline([('pre',pre_process_pipeline),('model',ElasticNet(alpha=0.001,l1_ratio=0.05))])
pred_log = cross_val_predict(pipe,X1,y_log,cv=5)
pred_price = np.expm1(pred_log)
rmse = np.sqrt(mean_squared_error(train['price'], pred_price))
print(rmse)

In [ ]:
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.exceptions import ConvergenceWarning

# Ignore specific ConvergenceWarnings
warnings.filterwarnings("ignore", category=ConvergenceWarning)

for name, model in [('OLS',LinearRegression()),
                    ('Ridge',Ridge(alpha=1.0)),
                    ('Lasso',Lasso(alpha=0.001)),
                    ('Elastic',ElasticNet(alpha=0.001,l1_ratio=0.05))] : 
    pipe = pipe = Pipeline([('pre',pre_process_pipeline),('model',model)])
    pred_log = cross_val_predict(pipe,X1,y_log,cv=5)
    pred_price = np.expm1(pred_log)
    rmse = np.sqrt(mean_squared_error(train['price'], pred_price))
    
    print(name , rmse)

In [ ]:
for col in train.select_dtypes(include=['category']).columns.tolist():
    print(train[col].unique().tolist())

In [ ]:
from sklearn.preprocessing import OrdinalEncoder


color_order = ['J', 'I', 'H', 'G', 'F', 'E', 'D']
cut_order = ['Fair', 'Good', 'Very Good', 'Premium', 'Ideal']
clarity_order = ['I1', 'SI2', 'SI1', 'VS2', 'VS1', 'VVS2', 'VVS1', 'IF']

num_feats_1 = ["carat",'depth','table','x','y','z','volume','density']
cat_feats = ['color','cut','clarity']
X1 = train[num_feats_1 + cat_feats].copy()

ordinal_enc = OrdinalEncoder(categories=[
    color_order,
    cut_order,
    clarity_order
])
pre_process_pipeline = ColumnTransformer([
    ('cat',OneHotEncoder(handle_unknown='ignore'),cat_feats),
    ('num',num_pipeline,num_feats)])
pre_process_pipeline_1 = ColumnTransformer([
    ('cat',ordinal_enc,cat_feats),
    ('num',num_pipeline,num_feats_1)])
pre_process_pipeline_2 = ColumnTransformer([
    ('cat',ordinal_enc,cat_feats),
    ('num',num_pipeline,num_feats)])
pre_process_pipeline_3 = ColumnTransformer([
    ('cat',OneHotEncoder(handle_unknown='ignore'),cat_feats),
    ('num',num_pipeline,num_feats_1)])

train_inputs = [
    (X,pre_process_pipeline,"num, pipe"),
    (X1,pre_process_pipeline_1,'num1 pipe1'),
    (X1,pre_process_pipeline_2,'num1 pipe'),
    (X1,pre_process_pipeline_3,'num pipe1'),
]

for X_train,pp_pipe,ip_name in train_inputs:
    for name, model in [('OLS',LinearRegression()),
                        ('Ridge',Ridge(alpha=1.0)),
                        ('Lasso',Lasso(alpha=0.001)),
                        ('Elastic',ElasticNet(alpha=0.001,l1_ratio=0.05))] : 
        pipe = Pipeline([('pre',pp_pipe),('model',model)])
        pred_log = cross_val_predict(pipe,X_train,y_log,cv=5)
        pred_price = np.expm1(pred_log)
        rmse = np.sqrt(mean_squared_error(train['price'], pred_price))
        
        print(ip_name ,name, rmse)

In [ ]:
X1.head()

In [ ]:
pre_process_pipeline_best = ColumnTransformer([
    ('cat',OneHotEncoder(handle_unknown='ignore'),cat_feats),
    ('num',num_pipeline,num_feats_1)])

pipe = Pipeline([('pre',pre_process_pipeline_best),('model',ElasticNet(alpha=0.001,l1_ratio=0.05))])
pred_log = cross_val_predict(pipe,X1,y_log,cv=5)
pred_price = np.expm1(pred_log)
rmse = np.sqrt(mean_squared_error(train['price'], pred_price))
print(rmse)

696 - lowest , non ordinal 